## Bibliotecas

In [1]:
import psycopg
import pandas as pd
import numpy as np

## Conexão

In [2]:
conn = psycopg.connect(
    host="localhost",
    port=5432,
    dbname="lh_nautical",
    user="postgres",
    password="Admin123"
)

In [3]:
# Busca no banco de dados os produtos comprados por cada cliente
query = """
SELECT DISTINCT
    o.customer_id,
    p.id AS product_id
FROM orders AS o
JOIN order_items AS oi
    ON oi.order_id = o.id
JOIN product_variants AS pv
    ON pv.id = oi.product_variant_id
JOIN products AS p
    ON p.id = pv.product_id
"""
df = pd.read_sql(query, conn)
df.head()

C:\Users\crisl\AppData\Local\Temp\ipykernel_4696\1165957718.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,customer_id,product_id
0,1404,249
1,1584,273
2,1499,304
3,1851,415
4,1412,441


## Matriz

In [5]:
matriz = df.pivot_table(
    index="customer_id",
    columns="product_id",
    aggfunc="size", # Conta a quantidade de vezes que cada cliente comprou cada produto
    fill_value=0  # Preenche com 0 os valores ausentes (quando o cliente não comprou o produto)
)
matriz.head()

product_id,1,2,3,4,5,6,7,8,9,10,...,491,492,493,494,495,496,497,498,499,500
customer_id,,,,,,,,,,,,,,,,,,,,,
1,0,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,1,0,...,0,0,1,0,0,0,0,0,0,0
3,0,0,0,0,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,1,0,1,1,0,1,...,0,1,0,0,1,0,0,0,0,0
5,0,0,1,0,0,1,0,0,1,0,...,0,1,0,0,0,0,1,0,0,0


In [6]:
matriz.head()

product_id,1,2,3,4,5,6,7,8,9,10,...,491,492,493,494,495,496,497,498,499,500
customer_id,,,,,,,,,,,,,,,,,,,,,
1,0,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,1,0,...,0,0,1,0,0,0,0,0,0,0
3,0,0,0,0,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,1,0,1,1,0,1,...,0,1,0,0,1,0,0,0,0,0
5,0,0,1,0,0,1,0,0,1,0,...,0,1,0,0,0,0,1,0,0,0


In [7]:
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
# T = .transpose() é usado para transpor a matriz, ou seja, trocar linhas por colunas e vice-versa. 
# Isso é necessário porque queremos calcular a similaridade entre produtos, e não entre clientes.
matriz.T

customer_id,1,2,3,4,5,6,7,8,9,10,...,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000
product_id,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,1,0,0,1,0,...,1,0,0,0,0,0,1,0,1,0
2,1,0,0,0,0,0,0,0,1,0,...,1,0,0,1,0,0,0,0,0,0
3,0,0,0,0,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,1,0,0,0,0,0,0,...,1,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
496,0,0,0,0,0,0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,1
497,0,0,0,0,1,0,0,1,1,1,...,0,0,1,0,1,0,0,1,0,1
498,0,0,0,0,0,0,0,1,1,0,...,0,0,0,1,0,0,0,0,0,0


# Similaridade

In [9]:
# Calcula a similaridade entre os produtos usando a métrica de similaridade do cosseno
similaridade = cosine_similarity(matriz.T)
similaridade

array([[1.        , 0.22667977, 0.16169042, ..., 0.22947837, 0.10497902,
        0.1509259 ],
       [0.22667977, 1.        , 0.17313298, ..., 0.18398784, 0.12803105,
        0.14725381],
       [0.16169042, 0.17313298, 1.        , ..., 0.18322532, 0.08409455,
        0.14579212],
       ...,
       [0.22947837, 0.18398784, 0.18322532, ..., 1.        , 0.0873679 ,
        0.16642902],
       [0.10497902, 0.12803105, 0.08409455, ..., 0.0873679 , 1.        ,
        0.12928748],
       [0.1509259 , 0.14725381, 0.14579212, ..., 0.16642902, 0.12928748,
        1.        ]], shape=(500, 500))

In [10]:
# Cria um DataFrame com a matriz de similaridade
similaridade_df = pd.DataFrame(
    similaridade,
    index=matriz.columns, # Define os rótulos das linhas como os IDs dos produtos
    columns=matriz.columns # Define os rótulos das colunas como os IDs dos produtos
)
similaridade_df.head()

product_id,1,2,3,4,5,6,7,8,9,10,...,491,492,493,494,495,496,497,498,499,500
product_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.226680,0.161690,0.145999,0.235263,0.194791,0.176263,0.113787,0.198287,0.127357,...,0.151973,0.150621,0.208979,0.120580,0.090764,0.160171,0.173645,0.229478,0.104979,0.150926
2,0.226680,1.000000,0.173133,0.189424,0.231390,0.203046,0.161226,0.128494,0.211475,0.094919,...,0.105911,0.159083,0.227560,0.122548,0.104824,0.189423,0.202057,0.183988,0.128031,0.147254
3,0.161690,0.173133,1.000000,0.120760,0.172844,0.173573,0.152272,0.109222,0.184563,0.090018,...,0.112532,0.135687,0.219806,0.100807,0.097202,0.157810,0.213454,0.183225,0.084095,0.145792
4,0.145999,0.189424,0.120760,1.000000,0.169926,0.151925,0.148931,0.080119,0.136513,0.091467,...,0.110063,0.145980,0.176573,0.108455,0.073203,0.169782,0.181804,0.164810,0.085152,0.156852
5,0.235263,0.231390,0.172844,0.169926,1.000000,0.188160,0.154636,0.134841,0.199603,0.105158,...,0.112039,0.131074,0.195149,0.116424,0.114970,0.183350,0.189613,0.221060,0.147357,0.183813


In [11]:
# Cria um DataFrame com a similaridade do produto com ID 180 (Motor de Popa 1949) em relação aos demais produtos
similaridade_motor = similaridade_df.loc[180]
similaridade_motor

product_id
1      0.233562
2      0.217172
3      0.166842
4      0.179499
5      0.184645
         ...   
496    0.188945
497    0.209201
498    0.193863
499    0.097910
500    0.198903
Name: 180, Length: 500, dtype: float64

In [12]:
similaridade_motor = similaridade_motor.drop(180)
similaridade_motor

product_id
1      0.233562
2      0.217172
3      0.166842
4      0.179499
5      0.184645
         ...   
496    0.188945
497    0.209201
498    0.193863
499    0.097910
500    0.198903
Name: 180, Length: 499, dtype: float64

# Ranking

In [14]:
# Cria um ranking dos produtos mais similares ao produto com ID 180
ranking = similaridade_motor.sort_values(ascending=False)
ranking

product_id
389    0.256553
295    0.256239
75     0.255785
337    0.239332
55     0.237744
         ...   
38     0.089076
209    0.088759
72     0.087527
45     0.084286
105    0.072630
Name: 180, Length: 499, dtype: float64

In [15]:
# Filtra os 5 produtos mais similares ao produto com ID 180
top5 = ranking.head(5)
print(top5)

product_id
389    0.256553
295    0.256239
75     0.255785
337    0.239332
55     0.237744
Name: 180, dtype: float64


In [16]:
# Busca no banco de dados os nomes dos produtos
query_produtos = """
SELECT
    id AS product_id,
    name AS product_name
FROM products;
"""
produtos = pd.read_sql(query_produtos, conn)
produtos.head()

C:\Users\crisl\AppData\Local\Temp\ipykernel_4696\3308662018.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  produtos = pd.read_sql(query_produtos, conn)


,product_id,product_name
0,1,Motor de Popa 6014
1,2,Vela Mestra 3870
2,3,Tinta Antifouling 5479
3,4,Sonar Transducer 9933
4,5,Bússola de Bordo 3641


In [17]:
top5_df = top5.reset_index() # Reseta o índice do DataFrame para que possamos fazer o merge com o DataFrame de produtos
top5_df.columns = [ # Renomeia as colunas
    "product_id",
    "similaridade"
]
top5_df

,product_id,similaridade
0,389,0.256553
1,295,0.256239
2,75,0.255785
3,337,0.239332
4,55,0.237744


In [18]:
# Faz o merge ranking de produtos mais similares com produtos para obter os nomes dos produtos
top5_df = top5_df.merge(
    produtos,
    on="product_id",
    how="left"
)
top5_df

,product_id,similaridade,product_name
0,389,0.256553,Motor de Popa 5331
1,295,0.256239,Cabo Náutico 2105
2,75,0.255785,Vela Mestra 1913
3,337,0.239332,Cabo Náutico 9048
4,55,0.237744,GPS Plotter 6249
